# Pipeline INTERSECT — LLM sinh answer + GIAO citation ∩ pool rerank

**Luồng:** Hybrid Retrieval → Reranking (pool-k) → **LLM SINH answer** → lấy **GIAO** giữa các điều LLM trích dẫn trong answer với pool rerank → derive `relevant_docs` TỪ các điều được chọn.

Khác với `fast-retrieval`:
- `answer` **KHÔNG rỗng** → chấm điểm 5 tiêu chí QA.
- `relevant_docs`/`relevant_articles` **nhất quán** (mọi văn bản đều có ≥1 Điều).
- Số lượng **biến thiên** (1..`--max-select`) theo những gì LLM thực sự trích.

- ⚠️ **CHẬM** (~29s/câu vì sinh answer đầy đủ) → 2000 câu ~16h. Dùng để nộp bản tính cả QA, không phải để dò nhanh.
- 💡 **Khuyến nghị:** chạy thử một file rút gọn (vd 50/100 câu) để kiểm tra pipeline trước, rồi mới chạy full 2000 câu để nộp.

In [ ]:
# 1) Cài dependencies (cần bitsandbytes/accelerate để nạp LLM 4-bit)
!pip install -q qdrant-client rank_bm25 underthesea sentence-transformers \
    transformers accelerate bitsandbytes python-dotenv tqdm

In [ ]:
# 2) Load secrets Qdrant từ Kaggle
import os
from kaggle_secrets import UserSecretsClient

try:
    secrets = UserSecretsClient()
    os.environ["QDRANT_URL"] = secrets.get_secret("QDRANT_URL")
    os.environ["QDRANT_API_KEY"] = secrets.get_secret("QDRANT_API_KEY")
    print("✅ Secrets loaded from Kaggle")
except Exception as e:
    print("⚠️ Không load được secrets:", e)

In [ ]:
# 3) Clone / pull code từ GitHub — NHÁNH test1 (main giữ nguyên cho mọi người)
import os, sys

GITHUB_USERNAME = "kimmttrung"
REPO_NAME = "legal-rag-vietnam"
BRANCH = "main"
REPO_URL = f"https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"
WORKING_DIR = f"/kaggle/working/{REPO_NAME}"

if os.path.exists(WORKING_DIR):
    print(f"🔄 Pull nhánh {BRANCH}...")
    !cd {WORKING_DIR} && git fetch origin {BRANCH} && git checkout {BRANCH} && git pull origin {BRANCH}
else:
    print(f"📥 Clone nhánh {BRANCH}...")
    !cd /kaggle/working && git clone -b {BRANCH} {REPO_URL}

if WORKING_DIR not in sys.path:
    sys.path.insert(0, WORKING_DIR)
print(f"✅ Đồng bộ hoàn tất! (branch={BRANCH})")

In [ ]:
# 4) Vào thư mục repo + kiểm tra file cần thiết
import os, sys

WORKING_DIR = "/kaggle/working/legal-rag-vietnam"
os.chdir(WORKING_DIR)
if sys.path[0] != WORKING_DIR:
    sys.path.insert(0, WORKING_DIR)

print("Working directory:", os.getcwd())
for f in ["fast_retrieval.py", "src/answer_intersect.py",
          "config/settings.py", "data/law_corpus_clean.json", "data/law_manifest.json"]:
    print(("✅ " if os.path.exists(f) else "❌ THIẾU: ") + f)

In [ ]:
# 5) Chọn nguồn câu hỏi (dataset đã upload lên Kaggle)
#    - Upload file R2AIStage1DATA.json (hoặc bản rút gọn N câu) lên Kaggle:
#         Add Input → Upload → Datasets → kéo file JSON vào → file sẽ nằm ở /kaggle/input/<ten-dataset>/...
#    - Muốn test bao nhiêu câu thì upload file chứa đúng bấy nhiêu câu (vd cắt còn 50/100 câu).
#    - Copy đường dẫn file trong /kaggle/input rồi DÁN vào INPUT_FILE bên dưới.
import glob, json

# >>> DÁN ĐƯỜNG DẪN FILE CÂU HỎI Ở ĐÂY <<<
INPUT_FILE = "/kaggle/input/r2aistage1data/R2AIStage1DATA.json"

# Nếu để None, ô này sẽ tự dò file JSON trong /kaggle/input để bạn copy đường dẫn đúng.
if INPUT_FILE is None or not os.path.exists(INPUT_FILE):
    cands = glob.glob("/kaggle/input/**/*.json", recursive=True)
    print("⚠️ Chưa trỏ đúng INPUT_FILE. Các file JSON tìm thấy trong /kaggle/input:")
    for c in cands:
        print("  -", c)
    INPUT_FILE = next((c for c in cands if any(k in c.lower()
                       for k in ["r2ai", "question", "data", "test"])), None) or (cands[0] if cands else None)

assert INPUT_FILE and os.path.exists(INPUT_FILE), \
    "Không tìm thấy INPUT_FILE! Hãy Add Input → upload dataset rồi dán đúng đường dẫn /kaggle/input/..."

# Chạy đúng số câu có trong file (không giới hạn) → upload bao nhiêu câu chạy bấy nhiêu.
NUM_ARG = ""
n_q = len(json.load(open(INPUT_FILE, encoding="utf-8")))
print(f"\n✅ INPUT_FILE = {INPUT_FILE} | số câu = {n_q}")

In [ ]:
# 6) Chạy pipeline INTERSECT (LLM sinh answer + giao citation∩pool)
#    Chỉnh --pool-k / --max-select để dò.
print(f"🚀 INTERSECT — {INPUT_FILE} {NUM_ARG}...\n")
!python fast_retrieval.py --input "{INPUT_FILE}" --output /kaggle/working/results.json \
    --llm-answer --pool-k 8 --max-select 5 {NUM_ARG}

In [ ]:
# 8) Kiểm tra + đóng gói submission.zip
import json, zipfile, statistics

results = json.load(open("/kaggle/working/results.json", encoding="utf-8"))
na = [len(r["relevant_articles"]) for r in results]
nd = [len(r["relevant_docs"]) for r in results]
has_ans = sum(1 for r in results if r["answer"].strip())
print(f"✅ results.json — {len(results)} bản ghi | answer khác rỗng: {has_ans}")
print("   id mẫu:", [r['id'] for r in results[:5]], "| kiểu:", type(results[0]['id']).__name__)
print(f"   TB điều/câu={statistics.mean(na):.2f} | TB văn bản/câu={statistics.mean(nd):.2f}")

with zipfile.ZipFile("/kaggle/working/submission.zip", "w", zipfile.ZIP_DEFLATED) as z:
    z.write("/kaggle/working/results.json", arcname="results.json")
print("✅ /kaggle/working/submission.zip (chứa results.json)")

print("\n----- Bản ghi mẫu -----")
print(json.dumps(results[0], ensure_ascii=False, indent=2)[:700])